<div style="background: linear-gradient(to bottom, #76C800, #006EC7); padding: 24px 28px; border-radius: 12px; color: white;">

# Prepare Training
...

</div>

## Dataset & Training Budget

| Parameter | Value |
|-----------|-------|
| Model | `Qwen/Qwen3-Coder-30B-A3B-Instruct` (30B total / 3B active, MoE) |
| Method | QLoRA — 4-bit NF4 base + LoRA adapters (rank 64, α 128) |
| Icons (after merge) | 216,209 |
| Augmentation | ~2.5× per icon → ~540K training records |
| Avg tokens / record | ~400 (≈30 caption + ≈370 SVG) |
| Tokens per epoch | ~216M |
| Target epochs | 3 |
| **Total tokens** | **~650M** |

In [ ]:
import math

TOTAL_TOKENS = 216_209 * 2.5 * 400 * 3  # icons × augmentation × avg_tokens × epochs

configs = {
    "1× H100 80GB":  8_000,
    "1× A100 80GB":  4_000,
    "4× H100 80GB": 32_000,
}

print(f"Total tokens to train: {TOTAL_TOKENS/1e6:.0f}M\n")
print(f"{'Config':<18} {'tok/s':>8} {'Hours':>8}")
print("-" * 38)
for label, tps in configs.items():
    hours = TOTAL_TOKENS / tps / 3600
    print(f"{label:<18} {tps:>8,} {hours:>8.1f}")

## GPU Provider Pricing (April 2026)

> On-demand rates. Spot/preemptible can be 40–70 % cheaper but risk interruption.

| Provider | GPU | $/hr (on-demand) | Notes |
|----------|-----|-----------------|-------|
| **Vast.ai** | H100 SXM | $1.55 – $1.99 | Marketplace; lowest floor; host-dependent reliability |
| **RunPod** | H100 SXM | $2.54 – $2.69 | Community cloud; per-minute billing; solid uptime |
| **Lambda Labs** | H100 SXM5 | $2.99 | Simple flat rate; no spot; good for long runs |
| **Google Cloud A3** | H100 | $3.00 | Per-GPU; 8-GPU bundles; aggressive 2025 price cuts |
| **Together.ai** | H100 (serverless) | $3.49 | No setup; billed per second; good for short jobs |
| **AWS P5** | H100 | $3.90 | After mid-2025 44 % cut; p5.48xlarge = 8 GPUs |
| **CoreWeave** | H100 PCIe | $4.76 | Enterprise SLAs; best for regulated workloads |
| **Modal** | H100 (serverless) | $4.76 | Per-second billing; easy infra; expensive per-hour |

**A100 80GB for reference:** $1.49 – $1.92/hr on competitive providers (RunPod, Vast.ai).
Slower (~4 k tok/s vs ~8 k tok/s) but ~40 % cheaper per hour.

In [ ]:
providers = [
    # (label,              gpu_count, tok_s,  usd_per_gpu_hr)
    ("Vast.ai",            1,  8_000,  1.99),
    ("RunPod",             1,  8_000,  2.69),
    ("Lambda Labs",        1,  8_000,  2.99),
    ("Google Cloud A3",    1,  8_000,  3.00),
    ("AWS P5",             1,  8_000,  3.90),
    ("CoreWeave",          1,  8_000,  4.76),
    ("RunPod A100",        1,  4_000,  1.49),
    ("Vast.ai 4×H100",     4, 32_000,  1.99),
    ("Lambda Labs 4×H100", 4, 32_000,  2.99),
]

print(f"{'Provider':<24} {'GPUs':>4} {'Hours':>7} {'Total $':>9}  GPU")
print("-" * 58)
for label, n_gpu, tps, rate in providers:
    hours = TOTAL_TOKENS / tps / 3600
    cost  = hours * rate * n_gpu
    gpu   = "H100" if "A100" not in label else "A100"
    print(f"{label:<24} {n_gpu:>4}  {hours:>6.1f}h  ${cost:>7.0f}   {gpu}")

## Recommendation

**Best value: RunPod H100 SXM (~$65 total)**
- Reliable uptime, per-minute billing, no spot interruptions
- ~27 hours wall-clock for 3 epochs

**Cheapest: Vast.ai H100 (~$54)**
- Marketplace model — vet the host (check uptime score, reviews)
- Risk: host can disconnect; save checkpoints every 500 steps (already in `06_train.sh`)

**Fastest iteration: Vast.ai 4×H100 (~$107, ~7 hours)**
- Worth it for hyperparameter sweeps; overkill for a single full run

**Avoid for this job:**
- A100 — similar cost to H100 but takes 2× longer; not worth it
- Serverless (Modal, Together.ai) — fine for inference, expensive for 27-hour training jobs
- CoreWeave / AWS — enterprise overhead, pay premium not justified for research

**Checkpoint strategy:** `06_train.sh` saves every 500 steps. On Vast.ai/RunPod,
also enable volume mounts so checkpoints survive instance termination.